# บทที่ 6 การจำแนกประเภทข้อมูลทางฟิสิกส์
โน้ตบุ๊กประกอบตำรา *วิทยาศาสตร์ข้อมูลและการเรียนรู้ของเครื่องสำหรับนักฟิสิกส์*
โค้ดทุกเซลล์คัดลอกมาจากสคริปต์ในโฟลเดอร์ `ch06_classification/` หัวเซลล์ **โค้ดที่ N.M** ตรงกับเลขในตำรา

**วิธีใช้**
1. รันเซลล์ **เตรียมโค้ด** ด้านล่างก่อนเสมอ (ทุกครั้งที่เปิดโน้ตบุ๊กหรือเริ่ม session ใหม่)
2. แต่ละหัวข้อเริ่มด้วยเซลล์ที่ขึ้นต้นด้วย `%reset -f` ซึ่งล้างตัวแปรเก่า ทำให้ผลเหมือนรันสคริปต์นั้นเดี่ยว ๆ
   ให้รันเซลล์ในหัวข้อเดียวกันเรียงจากบนลงล่าง (Shift+Enter) หรือใช้ Runtime → Run all
3. ใช้หน่วยประมวลผลกลาง (CPU) ตามค่าเริ่มต้น ไม่ต้องเปลี่ยนเป็น GPU
4. ไฟล์ที่สร้างขึ้นอยู่ใน `ch06_classification/outputs/` และหายเมื่อ session จบ ถ้าต้องการเก็บ ให้คัดลอกไปยัง Google Drive

ตัวเลขส่วนใหญ่ควรตรงกับตำรา การคำนวณที่ไวต่อการปัดเศษ (การฝึกโครงข่ายประสาทเทียม
และมอนติคาร์โลในพิกัดต่อเนื่อง) อาจต่างเล็กน้อยเมื่อรันบนเครื่องอื่น ดู `ch06_classification/README.md`


In [ ]:
# เตรียมโค้ด: ดึงโค้ดจาก GitHub (เมื่อรันบน Colab) และติดตั้งเฉพาะไลบรารีที่ยังไม่มี
import os, sys, subprocess, importlib.util
CHAPTER = 'ch06_classification'
REPO = 'https://github.com/busarapa-stack/dsml-for-physicists.git'
if 'google.colab' in sys.modules:
    ROOT = '/content/dsml-for-physicists'
    if not os.path.isdir(ROOT):
        subprocess.run(['git', 'clone', '-q', REPO, ROOT], check=True)
else:                                   # Jupyter บนเครื่องตนเอง: เปิดจากโฟลเดอร์ notebooks/
    ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(os.path.join(ROOT, CHAPTER))
os.makedirs('data', exist_ok=True); os.makedirs('outputs', exist_ok=True)
NEED = {'imblearn': 'imbalanced-learn>=0.12', 'matplotlib': 'matplotlib', 'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy', 'seaborn': 'seaborn>=0.12', 'sklearn': 'scikit-learn>=1.3'}
missing = [req for mod, req in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
print('โฟลเดอร์ทำงาน:', os.getcwd())
print('ติดตั้งเพิ่ม:', ', '.join(missing) if missing else 'ไม่มี')


## `01_higgs_toy_data.py` — โค้ดที่ 6.1, 6.2

Ch.6  Toy H->bb events vs background, and a stratified train/validation split.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 01_higgs_toy_data.py  จากโฟลเดอร์ของบท
__file__ = '01_higgs_toy_data.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.6  Toy H->bb events vs background, and a stratified train/validation split.

Book references: ssec:higgs-features, ssec:fs-PI, code:higgs-data, code:higgs-pipe (line for line)
All events are SYNTHETIC toy data (shapes illustrative only), seed 2026.
"""
import os
from pathlib import Path

os.chdir(Path(__file__).resolve().parent)
Path('data').mkdir(exist_ok=True)

### โค้ดที่ 6.1

In [ ]:
# --- code:higgs-data ---------------------------------------------------------
import numpy as np
import pandas as pd

def make_higgs_toy(n_sig=4000, n_bkg=16000, seed=2026):
    """Toy H->bb events vs background (GeV). Shapes are illustrative only."""
    rng = np.random.default_rng(seed)
    # signal: resonance at 125 GeV with ~10% resolution
    m_s   = rng.normal(125.0, 12.0, n_sig)
    pT_s  = rng.gamma(shape=4.0, scale=40.0, size=n_sig)     # mean 160 GeV
    dR_s  = 2 * m_s / np.maximum(pT_s, 60.0) * rng.normal(1.0, 0.15, n_sig)
    met_s = rng.exponential(20.0, n_sig)                       # semileptonic b
    # background: falling mass spectrum, softer pT, wider angles
    m_b   = 40.0 + rng.exponential(70.0, n_bkg)
    pT_b  = rng.gamma(shape=3.0, scale=40.0, size=n_bkg)     # mean 120 GeV
    dR_b  = rng.uniform(0.4, 4.0, n_bkg)
    met_b = rng.exponential(30.0, n_bkg)                       # e.g. ttbar
    df = pd.DataFrame({
        'm_bb':   np.concatenate([m_s, m_b]),
        'pT':     np.concatenate([pT_s, pT_b]),
        'dR':     np.clip(np.concatenate([dR_s, dR_b]), 0.4, 4.0),
        'ETmiss': np.concatenate([met_s, met_b]),
        'label':  np.concatenate([np.ones(n_sig, int), np.zeros(n_bkg, int)]),
    })
    return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

df = make_higgs_toy()

### โค้ดที่ 6.2

In [ ]:
# --- code:higgs-pipe ---------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

features = ['m_bb', 'pT', 'dR', 'ETmiss']
X = df[features].values
y = df['label'].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    df.to_csv('data/higgs_toy.csv', index=False)
    print(f"{len(df)} events, signal fraction: all {y.mean():.3f}, train {y_train.mean():.3f}, "
          f"validation {y_val.mean():.3f}; validation size {len(y_val)} "
          f"({int(y_val.sum())} signal, {int((1 - y_val).sum())} background)")
    print("\nmean of each feature (GeV, dR dimensionless):")
    print(df.groupby('label')[features].mean().round(2).rename(index={0: 'background', 1: 'signal'}))
    s = df[df.label == 1]
    print(f"signal m_bb: 95 % of events in [{s.m_bb.quantile(0.025):.0f}, {s.m_bb.quantile(0.975):.0f}] GeV; "
          f"background fraction below/above that window: "
          f"{(df[df.label == 0].m_bb < 101).mean():.2f} / {(df[df.label == 0].m_bb > 149).mean():.2f}")
    r = s.dR * s.pT / (2 * s.m_bb)
    print(f"signal dR*pT/(2 m_bb): median {r.median():.2f} (background {(df[df.label == 0].eval('dR*pT/(2*m_bb)')).median():.2f})")
    print("saved data/higgs_toy.csv")

## `02_logreg_forest.py` — โค้ดที่ 6.3, 6.4

Ch.6  Logistic regression and random forest pipelines; feature importance.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 02_logreg_forest.py  จากโฟลเดอร์ของบท
__file__ = '02_logreg_forest.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.6  Logistic regression and random forest pipelines; feature importance.

Book references: ssec:lr, ssec:dt-rf, tab:rf-imp, ssec:f1-imbalance (predict_proba notebox)
Listings reproduced line for line: code:lr, code:rf
"""
import os
from pathlib import Path

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
src = (HERE / '01_higgs_toy_data.py').read_text()
exec(src[src.index('\n# --- code:higgs-data'):src.index('\n# --- end of listings')])

### โค้ดที่ 6.3

In [ ]:
# --- code:lr -----------------------------------------------------------------
from sklearn.linear_model import LogisticRegression

lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=1.0, class_weight='balanced',
                                  random_state=42)),
])
lr_pipe.fit(X_train, y_train)
y_proba_lr = lr_pipe.predict_proba(X_val)[:, 1]
print(lr_pipe.named_steps['clf'].coef_)    # one weight per feature

### โค้ดที่ 6.4

In [ ]:
# --- code:rf -----------------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier

rf_pipe = Pipeline([
    ('scaler', StandardScaler()),      # harmless for trees; kept for uniformity
    ('clf',    RandomForestClassifier(n_estimators=200, max_depth=10,
                                      min_samples_leaf=10,
                                      class_weight='balanced',
                                      random_state=42, n_jobs=-1)),
])
rf_pipe.fit(X_train, y_train)
importance = rf_pipe.named_steps['clf'].feature_importances_

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    from sklearn.inspection import permutation_importance
    from sklearn.metrics import roc_auc_score

    perm = permutation_importance(rf_pipe, X_val, y_val, scoring='roc_auc',
                                  n_repeats=10, random_state=42, n_jobs=-1)
    print("\nfeature   impurity   permutation (drop in validation AUC)")
    for f, a, b in sorted(zip(features, importance, perm.importances_mean), key=lambda t: -t[1]):
        print(f"{f:7s}   {a:8.3f}   {b:8.3f}")
    print(f"sums: impurity {importance.sum():.2f}, permutation {perm.importances_mean.sum():.2f}")
    p_rf = rf_pipe.predict_proba(X_val)[:, 1]
    print(f"\nmean predict_proba of the forest on the validation set = {p_rf.mean():.3f} "
          f"(true signal fraction there {y_val.mean():.3f})")
    print(f"logistic regression validation AUC = {roc_auc_score(y_val, y_proba_lr):.3f}")

## `03_compare_five.py` — โค้ดที่ 6.5

Ch.6  Five classifiers on the toy Higgs data; confusion matrix; the |m_bb - 125| feature.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 03_compare_five.py  จากโฟลเดอร์ของบท
__file__ = '03_compare_five.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.6  Five classifiers on the toy Higgs data; confusion matrix; the |m_bb - 125| feature.

Book references: ssec:choose-cls, tab:cls-results, ssec:confusion, tab:metrics
Listing reproduced line for line: code:cls-compare
"""
import os
from pathlib import Path

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
for f in ('01_higgs_toy_data.py', '02_logreg_forest.py'):
    src = (HERE / f).read_text()
    exec(src[src.index('\n# --- code:'):src.index('\n# --- end of listings')])

### โค้ดที่ 6.5

In [ ]:
# --- code:cls-compare --------------------------------------------------------
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

classifiers = {
    'LogReg':     LogisticRegression(class_weight='balanced', random_state=42),
    'KNN':        KNeighborsClassifier(n_neighbors=25),
    'SVM (RBF)':  SVC(kernel='rbf', class_weight='balanced', random_state=42),
    'DecTree':    DecisionTreeClassifier(max_depth=8, class_weight='balanced',
                                         random_state=42),
    'RandForest': RandomForestClassifier(n_estimators=200, max_depth=10,
                                         min_samples_leaf=10,
                                         class_weight='balanced',
                                         random_state=42, n_jobs=-1),
}
for name, clf in classifiers.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', clf)])
    pipe.fit(X_train, y_train)
    score = (pipe.decision_function(X_val) if name == 'SVM (RBF)'
             else pipe.predict_proba(X_val)[:, 1])
    y_pred = pipe.predict(X_val)
    print(f"{name:11s} acc={accuracy_score(y_val, y_pred):.3f} "
          f"F1={f1_score(y_val, y_pred):.3f} AUC={roc_auc_score(y_val, score):.3f}")

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    from sklearn.metrics import confusion_matrix
    import numpy as np
    print(f"always 'background': accuracy {1 - y_val.mean():.3f}")

    # confusion matrix of the random forest of code:rf at the default threshold 0.5
    tn, fp, fn, tp = confusion_matrix(y_val, rf_pipe.predict(X_val)).ravel()
    print(f"\nrandom forest (code:rf): TN = {tn}, FP = {fp}, FN = {fn}, TP = {tp}")
    print(f"purity (precision) = {tp / (tp + fp):.3f}, efficiency (recall) = {tp / (tp + fn):.3f}, "
          f"specificity = {tn / (tn + fp):.3f}")
    P, R = 0.99, 0.05
    print(f"F1 for purity 0.99 and efficiency 0.05: {2 * P * R / (P + R):.3f} (arithmetic mean {(P + R) / 2:.2f})")

    # physics feature: distance from the Higgs mass
    X2_train, X2_val = X_train.copy(), X_val.copy()
    X2_train[:, 0] = np.abs(X2_train[:, 0] - 125.0)
    X2_val[:, 0] = np.abs(X2_val[:, 0] - 125.0)
    for name in ('LogReg', 'RandForest'):
        pipe = Pipeline([('scaler', StandardScaler()), ('clf', classifiers[name])])
        pipe.fit(X2_train, y_train)
        print(f"with |m_bb - 125| instead of m_bb: {name:10s} AUC = "
              f"{roc_auc_score(y_val, pipe.predict_proba(X2_val)[:, 1]):.3f}")

## `04_roc_significance.py` — โค้ดที่ 6.6

Ch.6  ROC curve, AUC and the threshold that maximises the discovery significance Z_A.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 04_roc_significance.py  จากโฟลเดอร์ของบท
__file__ = '04_roc_significance.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.6  ROC curve, AUC and the threshold that maximises the discovery significance Z_A.

Book references: ssec:roc-auc, ssec:roc-PI, E6.4, review question 6
Listing reproduced line for line: code:roc-sb
"""
import os
from pathlib import Path

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
for f in ('01_higgs_toy_data.py', '02_logreg_forest.py'):
    src = (HERE / f).read_text()
    exec(src[src.index('\n# --- code:'):src.index('\n# --- end of listings')])

### โค้ดที่ 6.6

In [ ]:
# --- code:roc-sb -------------------------------------------------------------
from sklearn.metrics import roc_curve, roc_auc_score

y_score = rf_pipe.predict_proba(X_val)[:, 1]
fpr, tpr, thresholds = roc_curve(y_val, y_score)
auc = roc_auc_score(y_val, y_score)

# expected yields in real data after preselection (NOT validation-set counts)
s_exp, b_exp = 50.0, 5000.0
S = s_exp * tpr                         # expected signal passing each cut
B = b_exp * fpr                         # expected background passing each cut

ok = B >= 10.0                          # avoid cuts that leave almost no background
Z_A = np.zeros_like(S)
Z_A[ok] = np.sqrt(2 * ((S[ok] + B[ok]) * np.log1p(S[ok] / B[ok]) - S[ok]))
i_opt = np.argmax(Z_A)
print(f"AUC = {auc:.3f}")
print(f"best cut {thresholds[i_opt]:.3f}: eff_S = {tpr[i_opt]:.3f}, "
      f"eff_B = {fpr[i_opt]:.4f}, S = {S[i_opt]:.1f}, B = {B[i_opt]:.1f}, "
      f"Z = {Z_A[i_opt]:.2f}")

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------


def z_asimov(S, B):
    return np.sqrt(2 * ((S + B) * np.log1p(S / B) - S))


if __name__ == '__main__':
    print(f"\nno cut: S/sqrt(B) = {s_exp / np.sqrt(b_exp):.2f}, Z_A = {z_asimov(s_exp, b_exp):.2f}")
    print(f"gain in significance from the classifier: x{Z_A[i_opt] / z_asimov(s_exp, b_exp):.1f}")
    purity_real = S[i_opt] / (S[i_opt] + B[i_opt])
    n_s_val, n_b_val = y_val.sum(), (1 - y_val).sum()
    purity_val = tpr[i_opt] * n_s_val / (tpr[i_opt] * n_s_val + fpr[i_opt] * n_b_val)
    print(f"purity at the best cut: validation set {purity_val:.3f}, expected real data {purity_real:.3f}; "
          f"S/B = {S[i_opt] / B[i_opt]:.2f}")

    # S/sqrt(B) instead of Z_A
    z_simple = np.zeros_like(S)
    z_simple[ok] = S[ok] / np.sqrt(B[ok])
    j = np.argmax(z_simple)
    print(f"maximising S/sqrt(B) instead: cut {thresholds[j]:.3f}, B = {B[j]:.1f}, S/sqrt(B) = {z_simple[j]:.2f}, "
          f"but Z_A there = {Z_A[j]:.2f} (< {Z_A[i_opt]:.2f})")

    # the wrong way: counts of the validation set itself
    Sv, Bv = n_s_val * tpr, n_b_val * fpr
    okv = Bv > 0
    zv = np.where(okv, Sv / np.sqrt(np.where(okv, Bv, 1)), 0)
    k = np.argmax(zv)
    print(f"S/sqrt(B) with validation-set counts (meaningless): max {zv[k]:.0f} at B = {Bv[k]:.0f} event(s)")

    # review question 6: purity when the true signal fraction is 1 %
    f = 0.01
    print(f"purity at f = 1 %: {tpr[i_opt] * f / (tpr[i_opt] * f + fpr[i_opt] * (1 - f)):.2f}")

    # E6.4: s_exp = 500
    for s_e in (50.0, 500.0, 2000.0):
        S2, B2 = s_e * tpr, b_exp * fpr
        Z2 = np.zeros_like(S2); Z2[ok] = z_asimov(S2[ok], B2[ok])
        m = np.argmax(Z2)
        print(f"s_exp = {s_e:5.0f}: best cut {thresholds[m]:.3f}, eff_S = {tpr[m]:.3f}, eff_B = {fpr[m]:.4f}, "
              f"S = {S2[m]:.0f}, B = {B2[m]:.0f}, S/B = {S2[m] / B2[m]:.2f}, Z_A = {Z2[m]:.1f}")

## `05_sdss_data.py` — โค้ดที่ 6.7, 6.8

Ch.6  The SDSS table used by code:sdss-cc and code:sdss-comp.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 05_sdss_data.py  จากโฟลเดอร์ของบท
__file__ = '05_sdss_data.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.6  The SDSS table used by code:sdss-cc and code:sdss-comp.

The book uses the 100,000-object table selected from SDSS DR17 and published for teaching as the
Kaggle "Stellar Classification Dataset - SDSS17" (fedesoriano).  Put the real file at
    data/raw/star_classification.csv
(see optional/README_sdss.md or run optional/fetch_sdss_skyserver.py) and this script only checks it.

If the real file is absent, this script writes a SYNTHETIC STAND-IN with the same columns and the same
class counts (59,445 galaxies, 21,594 stars, 18,961 quasars), so that every script of the chapter runs.
Its colours follow the physics described in the book only schematically (stellar locus, redder
galaxies, blue low-z quasars, z ~ 2.5-3 quasars on the stellar locus).  Numbers from the stand-in are
NOT results about the real sky; the book quotes no numbers from this table except the class fractions.
"""
import os
from pathlib import Path

import numpy as np
import pandas as pd

os.chdir(Path(__file__).resolve().parent)
RAW = Path('data/raw/star_classification.csv')
SOURCE = Path('data/raw/SOURCE.txt')
RAW.parent.mkdir(parents=True, exist_ok=True)


def make_standin(seed=2026):
    rng = np.random.default_rng(seed)
    n_gal, n_star, n_qso = 59445, 21594, 18961
    sc = lambda n, s: rng.normal(0, s, n)

    # stars: one temperature-like parameter t along the stellar locus (hot/blue t=0 -> cool/red t=1)
    t = rng.beta(2.0, 2.5, n_star)
    s_gr = -0.25 + 1.65 * t + sc(n_star, 0.05)
    # SDSS stellar locus: u-g stays near 1.0-1.2 for hot A/F stars (g-r < ~0.3), then climbs to ~2.6 for K/M
    s_ug = 1.05 + 1.6 * np.clip((s_gr - 0.25) / 0.95, 0, 1) + sc(n_star, 0.08)
    s_ri = 0.02 + 1.3 * t**2.2 + sc(n_star, 0.04)
    s_iz = 0.00 + 0.65 * t**2.2 + sc(n_star, 0.04)
    s_r = rng.uniform(14.0, 20.5, n_star)
    s_zred = rng.normal(0, 3e-4, n_star)

    # galaxies: red sequence and blue cloud, colours redden with redshift
    zg = np.clip(rng.gamma(3.0, 0.07, n_gal), 0.005, 1.0)
    red = rng.random(n_gal) < 0.55
    g_gr = np.where(red, 0.72 + 1.3 * zg, 0.42 + 0.9 * zg) + sc(n_gal, 0.10)
    g_ug = np.where(red, 1.75 + 0.8 * zg, 1.25 + 0.5 * zg) + sc(n_gal, 0.25)
    g_ri = 0.33 + 0.9 * zg + sc(n_gal, 0.08)
    g_iz = 0.22 + 0.35 * zg + sc(n_gal, 0.10)
    g_r = rng.uniform(15.0, 20.5, n_gal) + 2.0 * zg

    # quasars: blue in u-g until Lyman-alpha enters u (z > ~2.2), then u-g rises steeply
    zq = np.clip(np.concatenate([rng.normal(1.4, 0.6, int(0.7 * n_qso)),
                                 rng.normal(2.7, 0.35, n_qso - int(0.7 * n_qso))]), 0.1, 5.0)
    q_ug = 0.15 + 1.4 * np.clip(zq - 2.2, 0, None) + sc(n_qso, 0.18)
    q_gr = 0.15 + 0.25 * np.clip(zq - 2.4, 0, None) + 0.1 * np.sin(2.5 * zq) + sc(n_qso, 0.12)
    q_ri = 0.12 + 0.08 * np.sin(1.7 * zq) + sc(n_qso, 0.10)
    q_iz = 0.05 + 0.10 * np.cos(1.3 * zq) + sc(n_qso, 0.12)
    q_r = rng.uniform(17.0, 21.5, n_qso)

    def mags(ug, gr, ri, iz, r):
        g = r + gr; u = g + ug; i = r - ri; z = i - iz
        return u, g, r, i, z

    parts = []
    for cls, cols, zred in (('GALAXY', (g_ug, g_gr, g_ri, g_iz, g_r), zg),
                            ('STAR', (s_ug, s_gr, s_ri, s_iz, s_r), s_zred),
                            ('QSO', (q_ug, q_gr, q_ri, q_iz, q_r), zq)):
        u, g, r, i, z = mags(*cols)
        n = len(r)
        parts.append(pd.DataFrame({'alpha': rng.uniform(0, 360, n), 'delta': rng.uniform(-10, 70, n),
                                   'u': u, 'g': g, 'r': r, 'i': i, 'z': z,
                                   'class': cls, 'redshift': zred}))
    df = pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    df.insert(0, 'obj_ID', np.arange(len(df)) + 1_000_000_000)
    df.loc[df.sample(1, random_state=seed).index, ['u', 'g', 'z']] = -9999.0   # one sentinel row, as in the real table
    return df.round({'u': 5, 'g': 5, 'r': 5, 'i': 5, 'z': 5, 'redshift': 6})


def is_synthetic(path):
    """The stand-in has no spec_obj_ID column; the Kaggle file and the SkyServer fetch both have it."""
    return 'spec_obj_ID' not in pd.read_csv(path, nrows=1).columns


if __name__ == '__main__':
    if not RAW.exists() or is_synthetic(RAW):          # a real file is never overwritten
        make_standin().to_csv(RAW, index=False)
        SOURCE.write_text("SYNTHETIC stand-in written by 05_sdss_data.py (not real SDSS data)\n")
    else:
        SOURCE.write_text("REAL SDSS table supplied by the user (Kaggle SDSS17 or optional/fetch_sdss_skyserver.py)\n")
    df = pd.read_csv(RAW)
    print(f"source: {SOURCE.read_text().strip()}")
    print(f"{len(df)} rows; columns: {list(df.columns)}")
    frac = df['class'].value_counts(normalize=True)
    print("class fractions: " + ", ".join(f"{k} {v:.3f}" for k, v in frac.items()))
    bad = (df[['u', 'g', 'r', 'i', 'z']] < -100).any(axis=1).sum()
    print(f"rows with a sentinel magnitude (< -100): {bad}")

## `06_sdss_colors.py` — โค้ดที่ 6.7

Ch.6  Colour-colour diagrams of the SDSS table (real file or synthetic stand-in, see 05_sdss_data.py).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 06_sdss_colors.py  จากโฟลเดอร์ของบท
__file__ = '06_sdss_colors.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.6  Colour-colour diagrams of the SDSS table (real file or synthetic stand-in, see 05_sdss_data.py).

Book references: ssec:sdss-eda, code:sdss-cc (reproduced line for line; the figure is saved)
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import pandas as pd

os.chdir(Path(__file__).resolve().parent)
if not Path('data/raw/star_classification.csv').exists():
    os.system('python 05_sdss_data.py')

### โค้ดที่ 6.7

In [ ]:
# --- code:sdss-cc ------------------------------------------------------------
import matplotlib.pyplot as plt

df = pd.read_csv('data/raw/star_classification.csv')
bands = ['u', 'g', 'r', 'i', 'z']
df = df[(df[bands] > -100).all(axis=1)]          # drop sentinel magnitudes

df['u_g'] = df['u'] - df['g']                    # color indices remove distance
df['g_r'] = df['g'] - df['r']
df['r_i'] = df['r'] - df['i']
df['i_z'] = df['i'] - df['z']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for cls, color in [('STAR', 'tab:blue'), ('GALAXY', 'tab:red'), ('QSO', 'tab:green')]:
    sub = df[df['class'] == cls].sample(2000, random_state=1)
    axes[0].scatter(sub['u_g'], sub['g_r'], c=color, alpha=0.4, s=8, label=cls)
    axes[1].scatter(sub['g_r'], sub['r_i'], c=color, alpha=0.4, s=8, label=cls)
axes[0].set_xlabel('u - g'); axes[0].set_ylabel('g - r')
axes[1].set_xlabel('g - r'); axes[1].set_ylabel('r - i')
axes[0].set_xlim(-0.5, 4); axes[0].set_ylim(-0.5, 2.5)   # a few faint objects
axes[1].set_xlim(-0.5, 2.5); axes[1].set_ylim(-0.5, 1.5) # have extreme colours
axes[0].legend()

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    print(Path('data/raw/SOURCE.txt').read_text().strip())
    print(f"{len(df)} rows after dropping sentinel magnitudes")
    fig.savefig('outputs/ch06_sdss_color_color.png', dpi=110)
    print(df.groupby('class')[['u_g', 'g_r', 'r_i', 'i_z', 'redshift']].median().round(2))
    print("saved outputs/ch06_sdss_color_color.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `07_sdss_compare.py` — โค้ดที่ 6.8

Ch.6  Five classifiers on the three-class SDSS problem, macro-F1 by 5-fold CV;

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 07_sdss_compare.py  จากโฟลเดอร์ของบท
__file__ = '07_sdss_compare.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.6  Five classifiers on the three-class SDSS problem, macro-F1 by 5-fold CV;
then the same without the spectroscopic redshift (E6.5, notebox on data leakage).

Book references: ssec:sdss-cls, ssec:sdss-PI, E6.5, code:sdss-comp (reproduced line for line)
Uses the real table if present, otherwise the SYNTHETIC stand-in of 05_sdss_data.py.

    python 07_sdss_compare.py            # SVM trained on a random 10,000-row subset (book's advice)
    python 07_sdss_compare.py --full     # SVM on all 75,000 training rows (slow)
"""
import os
import sys
from pathlib import Path

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
src = (HERE / '06_sdss_colors.py').read_text()
exec(src[:src.index('\n# --- end of listings')].split('"""', 2)[2])      # code:sdss-cc (builds df)
FULL = '--full' in sys.argv
if not FULL:                              # the book: "subsample 10,000 training rows for SVM"
    from sklearn.svm import SVC as _SVC

    class SVC(_SVC):
        def fit(self, X, y, sample_weight=None):
            idx = np.random.default_rng(0).permutation(len(X))[:10_000]
            return super().fit(X[idx], y[idx])

### โค้ดที่ 6.8

In [ ]:
# --- code:sdss-comp ----------------------------------------------------------
from sklearn.model_selection import cross_val_score, StratifiedKFold

features = ['u_g', 'g_r', 'r_i', 'i_z', 'r', 'redshift']
X = df[features].values
y = df['class'].to_numpy()          # a plain NumPy array of labels
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

classifiers = {
    'LogReg':     LogisticRegression(max_iter=1000),
    'KNN':        KNeighborsClassifier(n_neighbors=15),
    'SVM (RBF)':  SVC(kernel='rbf', random_state=42),
    'DecTree':    DecisionTreeClassifier(max_depth=8, random_state=42),
    'RandForest': RandomForestClassifier(n_estimators=200, random_state=42,
                                         n_jobs=-1),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
results = {}
for name, clf in classifiers.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', clf)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1_macro')
    results[name] = (scores.mean(), scores.std())
    print(f"{name:11s} macro-F1 = {scores.mean():.3f} +/- {scores.std():.3f}")

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    print(Path('data/raw/SOURCE.txt').read_text().strip(), "| SVM on", "all rows" if FULL else "10,000 rows")
    labels = ['GALAXY', 'STAR', 'QSO']

    def report(feats, tag):
        Xa = df[feats].values
        Xtr, Xte, ytr, yte = train_test_split(Xa, y, test_size=0.25, stratify=y, random_state=42)
        rf = Pipeline([('scaler', StandardScaler()),
                       ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))]).fit(Xtr, ytr)
        yp = rf.predict(Xte)
        print(f"\n[{tag}] random forest, test set: macro-F1 = {f1_score(yte, yp, average='macro'):.3f}")
        cm = confusion_matrix(yte, yp, labels=labels)
        print("confusion matrix (rows true, columns predicted):", labels)
        print(cm)
        off = {f"{a}->{b}": cm[i, j] for i, a in enumerate(labels) for j, b in enumerate(labels) if i != j}
        print("largest confusions:", sorted(off.items(), key=lambda t: -t[1])[:3])
        imp = rf.named_steps['clf'].feature_importances_
        print("importance:", ", ".join(f"{f} {v:.2f}" for f, v in sorted(zip(feats, imp), key=lambda t: -t[1])))

    report(features, 'with redshift')
    no_z = [f for f in features if f != 'redshift']
    report(no_z, 'without redshift')
    from sklearn.model_selection import cross_val_score
    for name in ('LogReg', 'RandForest'):
        s = cross_val_score(Pipeline([('scaler', StandardScaler()), ('clf', classifiers[name])]),
                            df[no_z].values[:len(X)], y, cv=cv, scoring='f1_macro')
        print(f"without redshift, 5-fold CV on all rows: {name:10s} macro-F1 = {s.mean():.3f} +/- {s.std():.3f}")

# แนวคำตอบของแบบฝึกหัดที่ต้องรันโค้ด
ควรลองทำเองก่อน แล้วจึงรันเซลล์เหล่านี้เพื่อเทียบคำตอบ

## `exercises/E6_2_feature_matrix.py`

E6.2  Class fractions and the pair plot of the four toy Higgs features, coloured by class.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E6_2_feature_matrix.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E6_2_feature_matrix.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E6.2  Class fractions and the pair plot of the four toy Higgs features, coloured by class."""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import seaborn as sns

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
src = (HERE / '01_higgs_toy_data.py').read_text()
exec(src[src.index('\n# --- code:'):src.index('\n# --- end of listings')])

print(f"signal fraction: all {y.mean():.3f}, train {y_train.mean():.3f}, validation {y_val.mean():.3f}")
g = sns.pairplot(df.sample(3000, random_state=1), vars=features, hue='label',
                 plot_kws={'s': 6, 'alpha': 0.4}, diag_kind='hist')
g.savefig('outputs/E6_2_pairplot.png', dpi=90)
s = df[df.label == 1]
print(f"signal m_bb window (central 90 %): {s.m_bb.quantile(0.05):.0f}-{s.m_bb.quantile(0.95):.0f} GeV")
print(f"new feature dR*pT/(2 m_bb): signal median {(s.dR * s.pT / (2 * s.m_bb)).median():.2f}")
print("saved outputs/E6_2_pairplot.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `exercises/E6_3_timing_cv.py`

E6.3  Training and prediction time, and 5-fold stratified CV of F1, for the five classifiers.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E6_3_timing_cv.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E6_3_timing_cv.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E6.3  Training and prediction time, and 5-fold stratified CV of F1, for the five classifiers."""
import os
import time
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
for f in ('01_higgs_toy_data.py', '02_logreg_forest.py', '03_compare_five.py'):
    src = (HERE / f).read_text()
    exec(src[src.index('\n# --- code:'):src.index('\n# --- end of listings')])
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
print("\nmethod       fit (s)  predict (s)   CV F1 (mean +/- std)")
for name, clf in classifiers.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', clf)])
    t0 = time.perf_counter(); pipe.fit(X_train, y_train); t1 = time.perf_counter()
    pipe.predict(X_val); t2 = time.perf_counter()
    s = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1')
    print(f"{name:11s}  {t1 - t0:7.3f}  {t2 - t1:9.3f}     {s.mean():.3f} +/- {s.std():.3f}")

X2_train = X_train.copy(); X2_train[:, 0] = np.abs(X2_train[:, 0] - 125.0)
s = cross_val_score(Pipeline([('scaler', StandardScaler()), ('clf', classifiers['LogReg'])]),
                    X2_train, y_train, cv=cv, scoring='f1')
print(f"LogReg with |m_bb - 125|: CV F1 {s.mean():.3f} +/- {s.std():.3f}")
print("(times depend on the computer; compare their ratios)")

## `exercises/E6_4_roc_threshold_plots.py`

E6.4  ROC curve, Z_A versus threshold, and purity/efficiency versus threshold (expected real-data counts).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E6_4_roc_threshold_plots.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E6_4_roc_threshold_plots.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E6.4  ROC curve, Z_A versus threshold, and purity/efficiency versus threshold (expected real-data counts)."""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
src = (HERE / '04_roc_significance.py').read_text()
exec(src[src.index('for f in'):src.index("if __name__ == '__main__':")])

purity = np.where(S + B > 0, S / np.where(S + B > 0, S + B, 1), np.nan)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], 'k:'); ax[0].set_xlabel('background efficiency (FPR)')
ax[0].set_ylabel('signal efficiency (TPR)'); ax[0].set_title(f'ROC, AUC = {auc:.3f}')
sel = thresholds <= 1
ax[1].plot(thresholds[sel], Z_A[sel]); ax[1].axvline(thresholds[i_opt], ls=':')
ax[1].set_xlabel('threshold'); ax[1].set_ylabel(r'$Z_A$ (50 signal, 5000 background expected)')
ax[2].plot(thresholds[sel], purity[sel], label='purity (expected real data)')
ax[2].plot(thresholds[sel], tpr[sel], label='signal efficiency'); ax[2].axvline(thresholds[i_opt], ls=':')
ax[2].set_xlabel('threshold'); ax[2].legend()
fig.tight_layout(); fig.savefig('outputs/E6_4_roc_threshold.png', dpi=120)
print(f"\nbest threshold {thresholds[i_opt]:.3f}: eff_S {tpr[i_opt]:.3f}, eff_B {fpr[i_opt]:.4f}, "
      f"purity {purity[i_opt]:.3f}, Z_A {Z_A[i_opt]:.2f}")
print("the s_exp = 500 and 2000 cases are printed by 04_roc_significance.py")
print("saved outputs/E6_4_roc_threshold.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `exercises/E6_5_sdss_figures.py`

E6.5  Four colour-colour diagrams and the confusion matrices (with / without redshift) as figures.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E6_5_sdss_figures.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E6_5_sdss_figures.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E6.5  Four colour-colour diagrams and the confusion matrices (with / without redshift) as figures.
Numbers are printed by 07_sdss_compare.py.  Uses the real table if present, else the SYNTHETIC stand-in."""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
__file__ = str(HERE / '06_sdss_colors.py')          # the listing's os.chdir uses __file__
src = (HERE / '06_sdss_colors.py').read_text()
exec(src[:src.index('\n# --- end of listings')].split('"""', 2)[2])
print(Path('data/raw/SOURCE.txt').read_text().strip())

pairs = [('u_g', 'g_r'), ('g_r', 'r_i'), ('r_i', 'i_z'), ('u_g', 'r_i')]
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (a, b) in zip(axes, pairs):
    for cls, color in [('STAR', 'tab:blue'), ('GALAXY', 'tab:red'), ('QSO', 'tab:green')]:
        sub = df[df['class'] == cls].sample(2000, random_state=1)
        ax.scatter(sub[a], sub[b], c=color, s=4, alpha=0.3, label=cls)
    ax.set_xlabel(a.replace('_', ' - ')); ax.set_ylabel(b.replace('_', ' - '))
    ax.set_xlim(*df[a].quantile([0.005, 0.995])); ax.set_ylim(*df[b].quantile([0.005, 0.995]))  # ignore extreme colours
axes[0].legend(); fig.tight_layout(); fig.savefig('outputs/E6_5_color_color_4.png', dpi=90)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, feats, title in [(axes[0], ['u_g', 'g_r', 'r_i', 'i_z', 'r', 'redshift'], 'with redshift'),
                         (axes[1], ['u_g', 'g_r', 'r_i', 'i_z', 'r'], 'without redshift')]:
    Xtr, Xte, ytr, yte = train_test_split(df[feats].values, df['class'].to_numpy(), test_size=0.25,
                                          stratify=df['class'].to_numpy(), random_state=42)
    rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xtr, ytr)
    ConfusionMatrixDisplay.from_predictions(yte, rf.predict(Xte), labels=['GALAXY', 'STAR', 'QSO'],
                                            ax=ax, colorbar=False)
    ax.set_title(title)
fig.tight_layout(); fig.savefig('outputs/E6_5_confusion.png', dpi=110)
print("saved outputs/E6_5_color_color_4.png and outputs/E6_5_confusion.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `exercises/E6_6_smote_vs_weights.py`

E6.6  SMOTE versus class weights on a strongly imbalanced toy Higgs set (400 signal, 40,000 background).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E6_6_smote_vs_weights.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E6_6_smote_vs_weights.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E6.6  SMOTE versus class weights on a strongly imbalanced toy Higgs set (400 signal, 40,000 background).
Needs imbalanced-learn (pip install imbalanced-learn)."""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
src = (HERE / '01_higgs_toy_data.py').read_text()
exec(src[src.index('\n# --- code:higgs-data'):src.index('\n# --- code:higgs-pipe')])   # make_higgs_toy

df = make_higgs_toy(n_sig=400, n_bkg=40000)
features = ['m_bb', 'pT', 'dR', 'ETmiss']
X, y = df[features].values, df['label'].values
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
rf = dict(n_estimators=200, max_depth=10, min_samples_leaf=10, random_state=42, n_jobs=-1)

models = {
    'no correction': Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(**rf))]),
    'class_weight':  Pipeline([('scaler', StandardScaler()),
                               ('clf', RandomForestClassifier(class_weight='balanced', **rf))]),
    'SMOTE':         ImbPipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=42)),
                                  ('clf', RandomForestClassifier(**rf))]),
}
print("model           AUC    efficiency  purity   (threshold 0.5, validation set)")
for name, m in models.items():
    m.fit(X_train, y_train)
    p = m.predict_proba(X_val)[:, 1]
    yp = (p >= 0.5).astype(int)
    print(f"{name:14s}  {roc_auc_score(y_val, p):.3f}   {recall_score(y_val, yp):.3f}      "
          f"{precision_score(y_val, yp, zero_division=0):.3f}")

# synthetic SMOTE signal events vs real signal in the dR - pT plane
Xs = StandardScaler().fit(X_train)
Xr, yr = SMOTE(random_state=42).fit_resample(Xs.transform(X_train), y_train)
synth = Xs.inverse_transform(Xr[len(X_train):])            # SMOTE appends new samples at the end
real = X_train[y_train == 1]
rel = lambda A: A[:, 2] * A[:, 1] / (2 * A[:, 0])           # dR * pT / (2 m_bb), about 1 for real signal
print(f"\n{len(synth)} synthetic signal events; dR*pT/(2 m_bb): real signal median {np.median(rel(real)):.2f}, "
      f"IQR {np.subtract(*np.percentile(rel(real), [75, 25])):.2f}; synthetic median {np.median(rel(synth)):.2f}, "
      f"IQR {np.subtract(*np.percentile(rel(synth), [75, 25])):.2f}")
lo, hi = np.percentile(rel(real), [1, 99])
print(f"fraction outside the 1-99 % band of the real signal [{lo:.2f}, {hi:.2f}]: real 0.02 by construction, "
      f"synthetic {np.mean((rel(synth) < lo) | (rel(synth) > hi)):.3f}")
print("-> SMOTE joins close neighbours, so here the curved dR-pT relation survives almost intact;"
      " the danger grows when the minority class is sparse or the relation bends sharply")
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(synth[:3000, 1], synth[:3000, 2], s=3, alpha=0.3, label='SMOTE synthetic')
ax.scatter(real[:, 1], real[:, 2], s=6, c='k', label='real signal')
pt = np.linspace(60, 600, 200); ax.plot(pt, 250 / pt, 'r--', label=r'$2 m_H / p_T$')
ax.set_xlabel(r'$p_T$ (GeV)'); ax.set_ylabel(r'$\Delta R$'); ax.set_ylim(0, 4.2); ax.legend()
fig.tight_layout(); fig.savefig('outputs/E6_6_smote_dR_pT.png', dpi=150)
print("saved outputs/E6_6_smote_dR_pT.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `exercises/E6_7_importance_stability.py`

E6.7  Stability of random-forest feature importance over 20 bootstrap resamples of the training set.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E6_7_importance_stability.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E6_7_importance_stability.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E6.7  Stability of random-forest feature importance over 20 bootstrap resamples of the training set."""
import os
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
for f in ('01_higgs_toy_data.py', '02_logreg_forest.py'):
    src = (HERE / f).read_text()
    exec(src[src.index('\n# --- code:'):src.index('\n# --- end of listings')])

from sklearn.base import clone
rng = np.random.default_rng(2026)
imps = []
for b in range(20):
    idx = rng.integers(0, len(X_train), len(X_train))
    m = clone(rf_pipe).fit(X_train[idx], y_train[idx])
    imps.append(m.named_steps['clf'].feature_importances_)
imps = np.array(imps)
print("\nfeature   mean    std   (20 bootstrap resamples)")
for f, mu, sd in zip(features, imps.mean(0), imps.std(0, ddof=1)):
    print(f"{f:7s}  {mu:.3f}  {sd:.4f}")
d = imps[:, features.index('dR')] - imps[:, features.index('pT')]
print(f"dR - pT: mean {d.mean():.3f}, std {d.std(ddof=1):.3f}; dR above pT in {np.mean(d > 0) * 20:.0f} of 20 resamples")

## `exercises/E6_8_decision_boundaries.py`

E6.8  Decision boundaries of five classifiers on make_moons (500 points, noise 0.25).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E6_8_decision_boundaries.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E6_8_decision_boundaries.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E6.8  Decision boundaries of five classifiers on make_moons (500 points, noise 0.25)."""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

os.chdir(Path(__file__).resolve().parent.parent)
X, y = make_moons(n_samples=500, noise=0.25, random_state=2026)
models = {'LogReg': LogisticRegression(), 'KNN k=1': KNeighborsClassifier(1),
          'KNN k=25': KNeighborsClassifier(25), 'SVM (RBF)': SVC(kernel='rbf'),
          'DecTree': DecisionTreeClassifier(max_depth=6, random_state=42)}
xx, yy = np.meshgrid(np.linspace(-2, 3, 300), np.linspace(-1.5, 2, 300))
fig, axes = plt.subplots(1, 5, figsize=(18, 3.6))
for ax, (name, m) in zip(axes, models.items()):
    pipe = make_pipeline(StandardScaler(), m).fit(X, y)
    Z = pipe.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm'); ax.scatter(X[:, 0], X[:, 1], c=y, s=5, cmap='coolwarm')
    # count isolated "islands": connected regions of the minority prediction
    from scipy.ndimage import label
    n_regions = label(Z == 0)[1] + label(Z == 1)[1]
    ax.set_title(f"{name}\ntrain acc {pipe.score(X, y):.2f}, regions {n_regions}")
    print(f"{name:10s} training accuracy {pipe.score(X, y):.3f}, connected regions on the grid: {n_regions}")
fig.tight_layout(); fig.savefig('outputs/E6_8_decision_boundaries.png', dpi=110)
print("saved outputs/E6_8_decision_boundaries.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `exercises/E6_9_auc_bootstrap.py`

E6.9  Bootstrap confidence interval of the validation AUC of the random forest; AUCs of SVM and KNN.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E6_9_auc_bootstrap.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E6_9_auc_bootstrap.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E6.9  Bootstrap confidence interval of the validation AUC of the random forest; AUCs of SVM and KNN."""
import os
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
for f in ('01_higgs_toy_data.py', '02_logreg_forest.py'):
    src = (HERE / f).read_text()
    exec(src[src.index('\n# --- code:'):src.index('\n# --- end of listings')])
from sklearn.metrics import roc_auc_score

score = rf_pipe.predict_proba(X_val)[:, 1]
rng = np.random.default_rng(2026)
aucs = []
for _ in range(1000):
    i = rng.integers(0, len(y_val), len(y_val))
    aucs.append(roc_auc_score(y_val[i], score[i]))
lo, hi = np.percentile(aucs, [2.5, 97.5])
print(f"\nrandom forest AUC = {roc_auc_score(y_val, score):.3f}, bootstrap 95 % CI [{lo:.3f}, {hi:.3f}] "
      f"(std {np.std(aucs):.4f})")
print("SVM 0.977 and KNN 0.973 (tab:cls-results) lie", "inside" if lo <= 0.973 and 0.977 <= hi else "outside", "this interval")

## `exercises/E6_10_quasar_significance.py`

E6.10  Quasars as 'signal' (one-vs-rest), random forest without redshift, 200 expected quasars and

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E6_10_quasar_significance.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E6_10_quasar_significance.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E6.10  Quasars as 'signal' (one-vs-rest), random forest without redshift, 200 expected quasars and
20,000 other objects in the sky area of interest: the threshold that maximises Z_A."""
import os
from pathlib import Path

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
__file__ = str(HERE / '06_sdss_colors.py')          # the listing's os.chdir uses __file__
src = (HERE / '06_sdss_colors.py').read_text()
exec(src[:src.index('\n# --- end of listings')].split('"""', 2)[2])
print(Path('data/raw/SOURCE.txt').read_text().strip())

feats = ['u_g', 'g_r', 'r_i', 'i_z', 'r']
Xtr, Xte, ytr, yte = train_test_split(df[feats].values, df['class'].to_numpy(), test_size=0.25,
                                      stratify=df['class'].to_numpy(), random_state=42)
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xtr, ytr)
p_qso = rf.predict_proba(Xte)[:, list(rf.classes_).index('QSO')]
yq = (yte == 'QSO').astype(int)
fpr, tpr, thr = roc_curve(yq, p_qso)
s_exp, b_exp = 200.0, 20000.0
S, B = s_exp * tpr, b_exp * fpr
ok = B >= 10.0
Z = np.zeros_like(S)
Z[ok] = np.sqrt(2 * ((S[ok] + B[ok]) * np.log1p(S[ok] / B[ok]) - S[ok]))
i = np.argmax(Z)
print(f"QSO-vs-rest AUC = {roc_auc_score(yq, p_qso):.3f}")
print(f"best threshold {thr[i]:.3f}: efficiency {tpr[i]:.3f}, contamination rate {fpr[i]:.4f}, "
      f"S = {S[i]:.0f}, B = {B[i]:.0f}, purity {S[i] / (S[i] + B[i]):.2f}, Z_A = {Z[i]:.1f}")

# ข้อมูลจริง (ไม่บังคับ)
สคริปต์สำหรับดาวน์โหลดและอ่านข้อมูลจริงอยู่ใน `ch06_classification/optional/` วิธีดาวน์โหลดและผลที่ควรได้อยู่ใน `ch06_classification/README.md` ข้อมูลเหล่านี้ไม่ได้อยู่ใน GitHub เพราะขนาดใหญ่หรือมีเงื่อนไขสัญญาอนุญาต